# 🌸 Nayari Dataset Builder  v3
**Run locally.** Scans `dataset/` (including subdirectories) for `.md`, `.txt`, `.pdf`, and `.json` files, applies a robust multi-format parser, deduplicates, quality-filters, and exports `nayari_dataset.json`.  Then auto-uploads to Kaggle.

| Feature | Details |
|---|---|
| Speaker formats | `Name:`, `**Name**:`, `> Name:`, `[Name]:` |
| Scene splitting | Blank-line gaps, `---END---`, `<end>`, `===` dividers |
| Auto-aliases | Unknown speakers scanned from filenames & content |
| Quality filter | Min 2 turns, min 10 chars/message |
| Deduplication | MD5 fingerprint on normalised content |
| OOC stripping | `[OOC: …]` and `(OOC …)` blocks removed |
| Source tagging | Every conversation records its origin file |
| Token estimate | Rough GPT-style token count in the stats cell |


## 📦 Step 0 — Install & Import

In [1]:
# 1. INSTALL DEPENDENCIES
# Note: pathlib is a built-in Python module and doesn't need pip installation.
# Added 'kaggle' to support the auto-upload features in the final steps.
%pip install -U openai pdfplumber kaggle -q

import sys
import json
import time
import re
import pdfplumber
import warnings
import shutil
import subprocess
import hashlib
from pathlib import Path
from openai import OpenAI
from collections import defaultdict
from datetime import datetime, timezone

# Suppress PDF-specific extraction warnings for a cleaner console
warnings.filterwarnings("ignore", category=UserWarning)

print("✅ Environment initialized. All dependencies are ready.")


Note: you may need to restart the kernel to use updated packages.
✅ Environment initialized. All dependencies are ready.


## 1 — Character Details

In [1]:
import json
import re
from pathlib import Path
from openai import OpenAI

# --- 1. CORE CONFIGURATION ---
CONFIG = {
    "base_url": "http://127.0.0.1:1234/v1", 
    "api_key": "lm-studio",
    "model": "loaded-model"
}

client = OpenAI(base_url=CONFIG["base_url"], api_key=CONFIG["api_key"])

class Col:
    BLUE = "\033[94m"; CYAN = "\033[96m"; GREEN = "\033[92m"
    YELLOW = "\033[93m"; RED = "\033[91m"; BOLD = "\033[1m"; END = "\033[0m"

# --- 2. REPAIRED FUNCTIONS ---
def ask_llm(system_prompt, user_content):
    """Universal LLM caller that handles the 'Type' error by using text-mode."""
    try:
        response = client.chat.completions.create(
            model=CONFIG["model"],
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            # FIXED: Changed from 'json_object' to 'text' for maximum compatibility
            response_format={"type": "text"}, 
            temperature=0.2
        )
        
        content = response.choices[0].message.content
        
        # We manually find the JSON block in case the model adds chatter
        json_match = re.search(r'\{.*\}', content, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
        return None
        
    except Exception as e:
        print(f"{Col.RED}LLM Error: {e}{Col.END}")
        return None

def extract_character_profile(text):
    """Uses a strong prompt to ensure we still get JSON even in text-mode."""
    system_prompt = (
        "You are a Data Extractor. Extract Name, Type, Gender, and Personality "
        "from the text. Return ONLY a valid JSON object. Do not include any "
        "introductory text or explanation."
    )
    profile = ask_llm(system_prompt, text)
    return profile if isinstance(profile, dict) else {"name": "Unknown", "type": "Unknown", "personality": "Unknown"}

# --- 3. EXECUTION LOGIC ---
DATASET_DIR = Path("dataset")
identity_keywords = ["details", "bio", "profile", "identity", "char_info"]
details_candidates = []

if DATASET_DIR.exists():
    for f in DATASET_DIR.rglob("*"):
        if any(kw in f.name.lower() for kw in identity_keywords) and f.is_file():
            details_candidates.append(f)

if not details_candidates:
    print(f"{Col.YELLOW}⚠️ No identity files found in {DATASET_DIR}.{Col.END}")
    char_profile = {"name": "Unknown"}
else:
    target_file = details_candidates[0]
    print(f"{Col.GREEN}🔍 Analyzing Identity File: {target_file.name}{Col.END}")
    
    try:
        raw_content = target_file.read_text(encoding="utf-8", errors="replace")
        char_profile = extract_character_profile(raw_content)
        char_name = char_profile.get("name", "Unknown")
        
        print(f"{Col.BOLD}✨ Character Engine Initialized:{Col.END}")
        print(f"   Name: {Col.CYAN}{char_name}{Col.END} | Type: {char_profile.get('type', 'Unknown')}")
        print(f"   Personality: {char_profile.get('personality', 'Unknown')}\n")
    except Exception as e:
        print(f"{Col.RED}❌ Logic Error: {e}{Col.END}")



🔍 Analyzing Identity File: Nayari_Details.md
✨ Character Engine Initialized:
   Name: Unknown | Type: Unknown
   Personality: Unknown



## 2 — Robust Multi-Format Parser

Handles every speaker format seen in the repo:

```
Name: text           ← plain
**Name**: text       ← bold markdown
> Name: text         ← blockquote
[Name]: text         ← bracket
```

Scene splitting triggers on: `--- END ---`, `=== break ===`, `<end>`, or **3+ blank lines**.
OOC annotations `[OOC: …]` and `(OOC …)` are stripped from message content.

In [2]:
import json
import re
import time
import hashlib
from pathlib import Path
from collections import defaultdict

# --- 1. CONFIGURATION & REGEX (Your Logic) ---
BASE_USER_ALIASES      = {"me", "you", "tiaya"}
BASE_ASSISTANT_ALIASES = {"nayari", "nayri", "aura"}

SPEAKER_RE = re.compile(
    r"""^(?:\*{1,2}|\[|>\s*)?([A-Za-z][A-Za-z0-9 _\'\-]*) (?:\*{1,2}|\])? :\s*(.*)$""",
    re.VERBOSE | re.MULTILINE
)
END_RE = re.compile(r"^[-=*]{3,}\s*(<?(end|break|scene)>?)?\s*[-=*]{0,}$", re.I)
OOC_RE = re.compile(r"\[OOC:?[^\]]*\]|\(OOC[^)]*\)", re.IGNORECASE)

class Col:
    BLUE = "\033[94m"; CYAN = "\033[96m"; GREEN = "\033[92m"
    YELLOW = "\033[93m"; RED = "\033[91m"; BOLD = "\033[1m"; END = "\033[0m"

# --- 2. THE CORE FUNCTIONS (Your Parser Logic) ---

def _strip_ooc(text: str) -> str:
    return OOC_RE.sub("", text).strip()

def _inject_scene_ends(text: str) -> str:
    return re.sub(r"(\r?\n){3,}", "\n--- END ---\n", text)

def _detect_aliases(text: str):
    counts = defaultdict(int)
    for line in text.splitlines():
        m = SPEAKER_RE.match(line.strip())
        if m: counts[m.group(1).strip().lower()] += 1
    u_extra, a_extra = set(), set()
    asst_keywords = {"nayari", "nayri", "aura", "goddess"}
    for name, cnt in counts.items():
        if cnt < 2: continue
        if any(kw in name for kw in asst_keywords): a_extra.add(name)
        else: u_extra.add(name)
    return u_extra, a_extra

def parse_chat_text(text: str, filename: str = ""):
    u_extra, a_extra = _detect_aliases(text)
    user_aliases = BASE_USER_ALIASES | u_extra
    asst_aliases = BASE_ASSISTANT_ALIASES | a_extra
    
    text = _inject_scene_ends(text)
    lines, conversations, current_messages = text.splitlines(), [], []
    current_role, buf = None, []

    def flush():
        nonlocal buf
        if current_role and buf:
            content = _strip_ooc(" ".join(" ".join(buf).split()).strip())
            if len(content) >= 10:
                current_messages.append({"role": current_role, "content": content})
        buf = []

    def save():
        if len(current_messages) >= 2:
            conversations.append({"messages": list(current_messages), "source": filename})
        current_messages.clear()

    for line in lines:
        s = line.strip()
        if not s: continue
        if (s.startswith(("#", "-", "=")) and not SPEAKER_RE.match(s)) or END_RE.match(s):
            flush(); save(); current_role = None; continue
        m = SPEAKER_RE.match(s)
        if m:
            sp, rest = m.group(1).strip().lower(), m.group(2).strip()
            if sp in user_aliases: flush(); current_role = "user"; buf = [rest]
            elif sp in asst_aliases: flush(); current_role = "assistant"; buf = [rest]
            else: 
                if current_role: buf.append(s)
        elif current_role: buf.append(s)
    flush(); save()
    return conversations

# --- 3. THE EXECUTION LOOP (The Parser) ---

processed_data = {"conversations": [], "lore": [], "failed": []}
start_time = time.time()

# Ensure all_files exists (Discovery)
DATASET_DIR = Path("dataset") # Adjust this path as needed
all_files = sorted([f for f in DATASET_DIR.rglob("*") if f.suffix.lower() in {'.json', '.md', '.txt', '.pdf'} and f.is_file()])

print(f"\n{Col.BOLD}{Col.BLUE}🧠 Intelligence Engine: Analyzing {len(all_files)} files...{Col.END}\n")

for idx, path in enumerate(all_files, 1):
    rel = str(path.relative_to(DATASET_DIR))
    # Truncate long paths for the UI
    display_name = (rel[:37] + '..') if len(rel) > 40 else rel
    ext = path.suffix.lower()
    prefix = f"{Col.BOLD}[{idx:03}/{len(all_files)}]{Col.END}"
    
    try:
        # JSON PROCESSING
        if ext == ".json":
            data = json.loads(path.read_text(encoding="utf-8", errors="replace"))
            # Logic: is it lore?
            if "lore" in rel.lower() or "world" in rel.lower() or "details" in rel.lower():
                processed_data["lore"].append({"source": rel, "content": data})
                print(f"{prefix} 📜 {Col.YELLOW}LORE-JSON{Col.END}  | {display_name:<40} | {Col.YELLOW}Data Tree{Col.END}")
            else:
                convs = []
                if isinstance(data, list): convs = [{"messages": c["messages"], "source": rel} for c in data if "messages" in c]
                elif isinstance(data, dict):
                    if "conversations" in data: convs = data["conversations"]
                    elif "messages" in data: convs = [data]
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 💬 {Col.CYAN}CHAT-JSON{Col.END}  | {display_name:<40} | {Col.CYAN}{len(convs)} convs{Col.END}")

        # TEXT / MD PROCESSING
        elif ext in {".txt", ".md"}:
            raw_text = path.read_text(encoding="utf-8", errors="replace")
            # Logic: is it lore or chat?
            if "lore" in rel.lower() or "details" in rel.lower():
                processed_data["lore"].append({"source": rel, "text": raw_text})
                print(f"{prefix} 📝 {Col.YELLOW}TXT-LORE{Col.END}   | {display_name:<40} | {Col.YELLOW}{len(raw_text.split()):,} words{Col.END}")
            else:
                convs = parse_chat_text(raw_text, rel)
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 📝 {Col.CYAN}TXT-CHAT{Col.END}   | {display_name:<40} | {Col.CYAN}{len(convs)} scenes{Col.END}")

        # PDF PROCESSING
        elif ext == ".pdf":
            import pdfplumber
            with pdfplumber.open(path) as pdf:
                text = "\n".join([p.extract_text() or "" for p in pdf.pages])
            convs = parse_chat_text(text, rel)
            if convs:
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 📘 {Col.CYAN}PDF-CHAT{Col.END}   | {display_name:<40} | {Col.CYAN}{len(convs)} scenes{Col.END}")
            else:
                processed_data["lore"].append({"source": rel, "text": text})
                print(f"{prefix} 📘 {Col.YELLOW}PDF-LORE{Col.END}   | {display_name:<40} | {Col.YELLOW}Knowledge{Col.END}")

    except Exception as exc:
        print(f"{prefix} ❌ {Col.RED}FAILED{Col.END}     | {display_name:<40} | {Col.RED}{type(exc).__name__}{Col.END}")
        processed_data["failed"].append(rel)

# --- 4. FINAL SMART BREAKDOWN ---

total_convs = len(processed_data['conversations'])
total_msgs = sum(len(c.get("messages", [])) for c in processed_data["conversations"])
total_lore = len(processed_data['lore'])
elapsed = time.time() - start_time

print(f"\n{Col.BOLD}{'═'*75}{Col.END}")
print(f"{Col.BOLD}📊 INTELLIGENCE ENGINE: FINAL DATASET BREAKDOWN{Col.END}")
print(f"{Col.BOLD}{'═'*75}{Col.END}")
print(f"⏱️  Processing Time:   {Col.BOLD}{elapsed:.2f} seconds{Col.END}")
print(f"💬 Total Conversations: {Col.GREEN}{total_convs:,}{Col.END}")
print(f"💬 Total Messages:      {Col.GREEN}{total_msgs:,}{Col.END}")
print(f"📜 Lore/Knowledge:      {Col.YELLOW}{total_lore:,} entries{Col.END}")
print(f"❌ Failed Files:        {Col.RED}{len(processed_data['failed'])}{Col.END}")

if total_convs > 0:
    avg = total_msgs / total_convs
    print(f"📈 Content Density:     {Col.CYAN}{avg:.1f} messages per conversation{Col.END}")

print(f"{Col.BOLD}{'═'*75}{Col.END}")


🧠 Intelligence Engine: Analyzing 196 files...

[001/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_1.md    | 1 scenes
[002/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_10.md   | 4 scenes
[003/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_11.md   | 4 scenes
[004/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_12.md   | 4 scenes
[005/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_13.md   | 4 scenes
[006/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_14.md   | 4 scenes
[007/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_15.md   | 1 scenes
[008/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_16.md   | 1 scenes
[009/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_17.md   | 1 scenes
[010/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_18.md   | 1 scenes
[011/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_19.md   | 1 scenes
[012/196] 📝 TXT-CHAT   | set-1\md files\chats\Nayari_chat_2.md    | 1 scenes
[013/196] 📝 TXT-CHAT   | set

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


[129/196] 📘 PDF-LORE   | set-1\pdf files\lore\Discovery .pdf      | Knowledge
[130/196] 📘 PDF-LORE   | set-1\pdf files\lore\Her Beliefs .pdf    | Knowledge
[131/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy201.pdf        | Knowledge
[132/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy202.pdf        | Knowledge
[133/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy203.pdf        | Knowledge
[134/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy204.pdf        | Knowledge
[135/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy205.pdf        | Knowledge
[136/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy206.pdf        | Knowledge
[137/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy207.pdf        | Knowledge
[138/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy208.pdf        | Knowledge
[139/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy209.pdf        | Knowledge
[140/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy210.pdf        | Knowledge
[141/196] 📘 PDF-LORE   | set-1\pdf files\other\kegy211.pdf      

## 3 — Parse All Source Files (.md / .txt / .pdf / .json)

In [3]:
import json
import time
import re
from pathlib import Path

# --- 1. SETTINGS ---
# Set this to the EXACT name of your folder
FOLDER_NAME = "dataset\set-1" 
DATASET_DIR = Path(FOLDER_NAME)

# Extensions to look for (Case-Insensitive)
EXTENSIONS = {".md", ".txt", ".pdf", ".json"}

class Col:
    BLUE = "\033[94m"; CYAN = "\033[96m"; GREEN = "\033[92m"
    YELLOW = "\033[93m"; RED = "\033[91m"; BOLD = "\033[1m"; END = "\033[0m"

# --- 2. FOLDER AUDIT ---
print(f"{Col.BOLD}🔍 Folder Audit:{Col.END}")
if not DATASET_DIR.exists():
    print(f"  {Col.RED}❌ ERROR: Folder '{FOLDER_NAME}' not found!{Col.END}")
    print(f"  Current location: {Path.cwd()}")
    # List everything in current directory to help you find it
    print(f"  I see these folders here: {[d.name for d in Path('.').iterdir() if d.is_dir()]}")
else:
    # Check if the folder is empty
    all_items = list(DATASET_DIR.rglob("*"))
    print(f"  {Col.GREEN}✅ Folder found.{Col.END} Total items inside: {len(all_items)}")

# --- 3. RE-DEFINING MISSING HELPERS (To prevent NameErrors) ---
def is_lore_file(file_path):
    path_str = str(file_path).lower()
    return any(kw in path_str for kw in ['lore', 'world', 'bio', 'details', 'info'])

def parse_chat_text(text, source_name):
    # Matches Name: Message, **Name**: Message, etc.
    pattern = re.compile(r"^(?:\*?\*?|>?\s*\[?)([A-Za-z][A-Za-z0-9 _]*)(?:\]?|:|\*?\*?)\s*:\s*(.*)", re.MULTILINE)
    messages = []
    for m in pattern.finditer(text):
        messages.append({"role": m.group(1).strip(), "content": m.group(2).strip()})
    return [{"source": source_name, "messages": messages}] if messages else []

# --- 4. THE MAIN ENGINE ---
all_files = [f for f in DATASET_DIR.rglob("*") if f.suffix.lower() in EXTENSIONS and f.is_file()]

processed_data = {"conversations": [], "lore": [], "failed": []}
start_time = time.time()

print(f"\n{Col.BOLD}{Col.BLUE}🧠 Intelligence Engine: Analyzing {len(all_files)} files...{Col.END}\n")

for idx, path in enumerate(all_files, 1):
    rel = str(path.relative_to(DATASET_DIR))
    ext = path.suffix.lower()
    prefix = f"{Col.BOLD}[{idx}/{len(all_files)}]{Col.END}"
    
    try:
        # JSON
        if ext == ".json":
            data = json.loads(path.read_text(encoding="utf-8", errors="replace"))
            if is_lore_file(path):
                processed_data["lore"].append({"source": rel, "content": data})
                print(f"{prefix} 📜 {Col.YELLOW}LORE-JSON{Col.END}  | {rel[:40]}")
            else:
                processed_data["conversations"].append({"messages": data, "source": rel})
                print(f"{prefix} 💬 {Col.CYAN}CHAT-JSON{Col.END}  | {rel[:40]}")

        # TEXT/MD
        elif ext in {".md", ".txt"}:
            raw_text = path.read_text(encoding="utf-8", errors="replace")
            if is_lore_file(path):
                processed_data["lore"].append({"source": rel, "text": raw_text})
                print(f"{prefix} 📝 {Col.YELLOW}TXT-LORE{Col.END}   | {rel[:40]}")
            else:
                convs = parse_chat_text(raw_text, rel)
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 📝 {Col.CYAN}TXT-CHAT{Col.END}   | {rel[:40]}")

    except Exception as e:
        print(f"{prefix} ❌ {Col.RED}FAILED{Col.END}     | {rel[:40]} | {str(e)[:30]}")
        processed_data["failed"].append(rel)

# --- 5. SUMMARY ---
elapsed = time.time() - start_time
print(f"\n{Col.BOLD}📊 FINAL DATASET REPORT{Col.END}")
print(f"══════════════════════════════════════════")
print(f"⏱️  Time:        {elapsed:.2f}s")
print(f"💬 Dialogues:   {Col.GREEN}{len(processed_data['conversations'])}{Col.END}")
print(f"📜 Lore:        {Col.YELLOW}{len(processed_data['lore'])}{Col.END}")
print(f"❌ Failed:      {Col.RED}{len(processed_data['failed'])}{Col.END}")
print(f"══════════════════════════════════════════")

🔍 Folder Audit:
  ✅ Folder found. Total items inside: 203

🧠 Intelligence Engine: Analyzing 196 files...

[69/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_1.md
[70/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_10.md
[71/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_11.md
[72/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_12.md
[73/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_13.md
[74/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_14.md
[75/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_15.md
[76/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_16.md
[77/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_17.md
[78/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_18.md
[79/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_19.md
[80/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_2.md
[81/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_20.md
[82/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_21.md
[83/196] 📝 TXT-CHAT   | md files\chats\Nayari_chat_22.md
[84/196] 📝 TXT-CHAT   | md files\chats\Na

<>:8: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:8: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\UTiaya\AppData\Local\Temp\ipykernel_7708\1183355164.py:8: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  FOLDER_NAME = "dataset\set-1"


## 4 — Lore → Training Conversations

Each PDF lore section is chunked into paragraphs.
Each chunk is paired with a varied natural-language prompt so the model learns
to recall Nayari's lore organically. Multiple prompts per chunk for augmentation.

In [5]:
import json
import time
import re
from pathlib import Path
from openai import OpenAI  # pip install openai

# --- PROVIDER CONFIGURATION ---
# For Ollama: "http://localhost:11434/v1", api_key="ollama"
# For LM Studio: "http://localhost:1234/v1", api_key="lm-studio"
# For OpenAI and Other APIs: "https://api.openai.com/v1", api_key="sk-..."

PROVIDER_CONFIG = {
    "base_url": "http://127.0.0.1:1234/v1", # Update to the base url 
    "api_key": "lm-studio",
    "model": "loaded-model" 
}

client = OpenAI(
    base_url=PROVIDER_CONFIG["base_url"],
    api_key=PROVIDER_CONFIG["api_key"]
)

class Col:
    BLUE, CYAN, GREEN, YELLOW, RED, BOLD, END = "\033[94m", "\033[96m", "\033[92m", "\033[93m", "\033[91m", "\033[1m", "\033[0m"

# --- LLM CORE LOGIC ---

def call_llm(system_msg, user_msg):
    """Universal wrapper for any OpenAI-compatible API."""
    try:
        response = client.chat.completions.create(
            model=PROVIDER_CONFIG["model"],
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}
            ],
            response_format={"type": "json_object"}, # Supported by most modern models
            temperature=0.1
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"  {Col.RED}LLM Error: {e}{Col.END}")
        return None

# --- PROCESSING TASKS ---

def process_lore_with_llm(chunk, source_name):
    """Turns dry lore into a Q&A pair."""
    system_prompt = """
    You are a Dataset Architect. Convert the provided LORE text into a single Conversation Pair.
    1. Clean any PDF/OCR artifacts.
    2. Create a 'user' question that naturally asks for this information.
    3. The 'assistant' response should be the cleaned text.
    Return ONLY JSON: {"user": "...", "assistant": "..."}
    """
    result = call_llm(system_prompt, chunk)
    if result:
        return {
            "messages": [
                {"role": "user", "content": result["user"]},
                {"role": "assistant", "content": result["assistant"]}
            ],
            "metadata": {"source": source_name, "type": "synthetic_lore"}
        }
    return None

def process_chat_with_llm(raw_chat, source_name):
    """Normalizes messy chat transcripts into structured JSON."""
    system_prompt = """
    Convert this messy chat transcript into a structured JSON list of messages.
    Normalize names (e.g., 'Nayari_123' -> 'Nayari'). 
    Return ONLY JSON: {"messages": [{"role": "name", "content": "..."}, ...]}
    """
    result = call_llm(system_prompt, raw_chat)
    if result:
        # Add source metadata to the returned conversation
        res = {"messages": result["messages"], "metadata": {"source": source_name, "type": "cleaned_chat"}}
        return res
    return None

# --- MAIN LOOP ---

def intelligence_engine(all_files):
    processed_dataset = []
    
    print(f"\n{Col.BOLD}{Col.BLUE}🧠 Intelligence Engine Active ({PROVIDER_CONFIG['model']}){Col.END}\n")

    for idx, path in enumerate(all_files, 1):
        rel = str(path.name)
        ext = path.suffix.lower()
        print(f"[{idx}/{len(all_files)}] Processing {Col.CYAN}{rel}{Col.END}...")

        try:
            if ext in [".md", ".txt"]:
                content = path.read_text(encoding="utf-8", errors="replace")
                
                # Heuristic to decide which LLM prompt to use
                if "lore" in rel.lower() or len(re.findall(r'\b(is|was|the|history)\b', content[:200])) > 5:
                    # Treat as Lore: Chunk it if long
                    chunks = [content[i:i+1500] for i in range(0, len(content), 1500)]
                    for chunk in chunks:
                        data = process_lore_with_llm(chunk, rel)
                        if data: processed_dataset.append(data)
                else:
                    # Treat as Chat
                    data = process_chat_with_llm(content, rel)
                    if data: processed_dataset.append(data)

            elif ext == ".json":
                # For JSON, we might just want to clean it up or verify it
                raw_json = path.read_text(encoding="utf-8")
                # You can either append directly or run through LLM to 'normalize' 
                processed_dataset.append({"raw_json": json.loads(raw_json), "metadata": {"source": rel}})

        except Exception as e:
            print(f"  {Col.RED}Failed file {rel}: {e}{Col.END}")

    return processed_dataset

# --- EXECUTION ---
# DATASET_DIR = Path("./your_data")
# all_files = list(DATASET_DIR.rglob("*.*"))
# final_data = intelligence_engine(all_files)

# Save results
# with open("formatted_dataset.jsonl", "w") as f:
#     for entry in final_data:
#         f.write(json.dumps(entry) + "\n")

## 5 — Deduplication & Quality Filter

In [6]:
import hashlib
import json

# --- 1. MEMORY-EFFICIENT DEDUPLICATION ---
def smart_deduplicate(conversations):
    """
    Uses SHA-256 hashing to find identical content across thousands of entries 
    without crashing your 8GB RAM.
    """
    unique_convs = []
    seen_hashes = set()
    
    for conv in conversations:
        # Create a unique fingerprint based on message content
        content_str = "".join([m["content"] for m in conv["messages"]])
        fingerprint = hashlib.sha256(content_str.encode('utf-8')).hexdigest()
        
        if fingerprint not in seen_hashes:
            unique_convs.append(conv)
            seen_hashes.add(fingerprint)
            
    return unique_convs

# --- 2. LLM-POWERED MESSAGE REPAIR ---
def repair_oversized_message(role, content, char_name):
    """
    Instead of just flagging, ask the Small LLM to break it down.
    This fixes PDF 'walls of text' into natural dialogue.
    """
    repair_prompt = f"""
    This message is too long for training. 
    If it's Lore: Summarize it into a concise paragraph.
    If it's Dialogue: Split it into 3 shorter messages.
    Return ONLY JSON: {{"repaired_content": "..."}}
    """
    
    # We only call the LLM if the message is actually a problem
    # This saves your Iris Xe from overworking
    result = call_small_llm(repair_prompt, content[:2000])
    return result.get("repaired_content", content) if result else content

# --- 3. THE FINAL ARCHITECT REPORT ---

def finalize_dataset(processed_data, char_name):
    print(f"\n{Col.BOLD}🛠️  Refining & Optimizing Dataset...{Col.END}")
    
    # Smart Grouping
    all_raw = processed_data['conversations']
    lore_raw = processed_data['lore']
    
    # Deduplicate early to save memory
    initial_count = len(all_raw) + len(lore_raw)
    optimized_convs = smart_deduplicate(all_raw + lore_raw)
    final_count = len(optimized_convs)
    
    print(f"  {Col.GREEN}✓{Col.END} Deduplication: {initial_count} -> {final_count} (Removed {initial_count - final_count})")

    # Smart Sanity Check & Auto-Repair
    WARN_CHARS = 2500 
    repaired_count = 0
    
    for conv in optimized_convs:
        for m in conv["messages"]:
            if len(m["content"]) > WARN_CHARS:
                print(f"  {Col.YELLOW}⚡ Auto-Repairing long message ({len(m['content'])} chars)...{Col.END}")
                m["content"] = repair_oversized_message(m["role"], m["content"], char_name)
                repaired_count += 1

    # Final Categorization for stats
    stats = {
        "md": len([c for c in optimized_convs if str(c.get('source','')).endswith('.md')]),
        "pdf": len([c for c in optimized_convs if str(c.get('source','')).endswith('.pdf')]),
        "json": len([c for c in optimized_convs if str(c.get('source','')).endswith('.json')]),
        "txt": len([c for c in optimized_convs if str(c.get('source','')).endswith('.txt')])
    }

    print(f"\n{Col.BOLD}📊 SMART DATASET SUMMARY{Col.END}")
    print(f"══════════════════════════════════════════")
    print(f"  Total Training Pairs: {Col.CYAN}{len(optimized_convs)}{Col.END}")
    print(f"  Auto-Repaired Items:  {Col.YELLOW}{repaired_count}{Col.END}")
    print(f"  Source Diversity:")
    for ext, count in stats.items():
        if count > 0:
            print(f"    - {ext.upper()}: {count}")
    print(f"══════════════════════════════════════════")
    
    return optimized_convs

# --- EXECUTION ---
# final_dataset = finalize_dataset(processed_data, char_name)

## 6 — Preview & Statistics Dashboard

In [7]:
import hashlib
from collections import defaultdict

def get_content_hash(messages):
    """Creates a unique fingerprint for a conversation to catch duplicates."""
    full_str = "".join([f"{m['role']}:{m['content']}" for m in messages])
    return hashlib.sha256(full_str.encode('utf-8')).hexdigest()

def smart_optimize_dataset(processed_data, char_name):
    print(f"\n{Col.BOLD}{Col.BLUE}🛠️  Running Smart Optimizer...{Col.END}")
    
    # 1. Automatic Categorization by Source Extension
    all_raw = processed_data.get('conversations', []) + processed_data.get('lore', [])
    by_ext = defaultdict(list)
    
    for conv in all_raw:
        ext = Path(conv.get('source', 'unknown.txt')).suffix.lower() or '.txt'
        by_ext[ext].append(conv)

    # 2. Memory-Efficient Deduplication (Hash-based)
    unique_convs = []
    seen_hashes = set()
    duplicates_removed = 0

    for conv in all_raw:
        h = get_content_hash(conv.get('messages', []))
        if h not in seen_hashes:
            unique_convs.append(conv)
            seen_hashes.add(h)
        else:
            duplicates_removed += 1

    print(f"  {Col.GREEN}✓{Col.END} Deduplication complete: {Col.CYAN}{duplicates_removed}{Col.END} duplicates purged.")

    # 3. Smart Length Management (Thresholding & Repair)
    MAX_DIALOGUE_LENGTH = 2800  # Threshold for "Too Long"
    final_cleaned = []
    repair_count = 0

    print(f"  {Col.GREEN}✓{Col.END} Checking message health...")
    
    for conv in unique_convs:
        is_healthy = True
        new_messages = []
        
        for msg in conv.get("messages", []):
            content_len = len(msg["content"])
            
            # If a message is massive, it's likely a PDF dump or Lore block
            if content_len > MAX_DIALOGUE_LENGTH:
                # SMART REPAIR: Use your LM Studio model to summarize/split
                print(f"    {Col.YELLOW}⚡ Repairing oversized msg ({content_len} chars) in {conv.get('source')}{Col.END}")
                
                repair_prompt = f"The following text is too long for a chat message. Please summarize it into 2-3 concise paragraphs for {char_name}."
                repaired = call_small_llm(repair_prompt, msg["content"][:4000]) # Cap input for RAM safety
                
                if repaired and "repaired_content" in repaired:
                    msg["content"] = repaired["repaired_content"]
                    repair_count += 1
                else:
                    # Fallback: Just truncate if LLM fails
                    msg["content"] = msg["content"][:MAX_DIALOGUE_LENGTH] + "... [Truncated]"
            
            new_messages.append(msg)
        
        conv["messages"] = new_messages
        final_cleaned.append(conv)

    # 4. Final Dataset Health Report
    print(f"\n{Col.BOLD}📊 INTELLIGENCE REPORT: {char_name}{Col.END}")
    print(f"═" * 45)
    print(f"  Total Conversations:  {Col.GREEN}{len(final_cleaned)}{Col.END}")
    print(f"  Duplicates Removed:   {Col.RED}{duplicates_removed}{Col.END}")
    print(f"  Messages Repaired:    {Col.YELLOW}{repair_count}{Col.END}")
    print(f"  Source Breakdown:")
    for ext, items in by_ext.items():
        print(f"    - {ext.upper():5}: {len(items)} files")
    print(f"═" * 45)
    
    return final_cleaned

# --- EXECUTION ---
# all_conversations = smart_optimize_dataset(processed_data, char_name)

## 7 — Export JSON

In [11]:
import json
from datetime import datetime, timezone

def save_smart_dataset(all_conversations, char_profile, output_path, model_name):
    """
    Finalizes the dataset with metadata, quality metrics, and character identity.
    """
    print(f"\n{Col.BOLD}{Col.BLUE}💾 Generating Final Intelligence Package...{Col.END}")

    # 1. Dynamic Source Analysis
    # Instead of manual counts, we scan the sources present in the final list
    source_stats = {}
    total_msgs = 0
    
    for conv in all_conversations:
        # Count sources
        src = conv.get("metadata", {}).get("source", "unknown")
        ext = src.split('.')[-1].lower() if '.' in src else 'other'
        source_stats[ext] = source_stats.get(ext, 0) + 1
        
        # Count total messages for the report
        total_msgs += len(conv.get("messages", []))

    # 2. Quality Metrics (The "Smart" Part)
    avg_turns = total_msgs / len(all_conversations) if all_conversations else 0
    
    # Estimate tokens (Roughly 1 token ≈ 4 chars for English)
    total_chars = sum(len(m["content"]) for c in all_conversations for m in c["messages"])
    est_tokens = total_chars // 4

    # 3. Construct the Final JSON
    dataset_json = {
        "dataset_metadata": {
            "version": "3.0-smart",
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "engine": model_name,
            "stats": {
                "total_conversations": len(all_conversations),
                "total_messages": total_msgs,
                "avg_turns_per_conv": round(avg_turns, 2),
                "estimated_tokens": est_tokens,
                "source_distribution": source_stats
            }
        },
        "character_profile": {
            "identity": {
                "name": char_profile.get("name"),
                "type": char_profile.get("type"),
                "gender": char_profile.get("gender")
            },
            "personality": {
                "traits": char_profile.get("traits"),
                "description": char_profile.get("personality")
            },
            "lore_data": char_profile.get("lore_sections", [])
        },
        "conversations": all_conversations
    }

    # 4. Memory-Safe Write
    try:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(dataset_json, f, ensure_ascii=False, indent=2)
            
        file_size_mb = output_path.stat().st_size / (1024 * 1024)
        
        # 5. Final Architect's Success Report
        print(f"\n{Col.GREEN}✅ DATASET ARCHITECT SUCCESS{Col.END}")
        print(f"═" * 45)
        print(f"  📂 File:       {output_path.resolve()}")
        print(f"  ⚖️  Size:       {file_size_mb:.2f} MB")
        print(f"  💬 Dialogues:  {len(all_conversations)}")
        print(f"  🔢 Est Tokens: {est_tokens:,}")
        print(f"  📊 Diversity:  {', '.join([f'{k.upper()}: {v}' for k, v in source_stats.items()])}")
        
        # Smart Warning for Fine-tuning
        if est_tokens < 50000:
            print(f"\n{Col.YELLOW}💡 Pro-Tip: Your dataset is a bit small (<50k tokens). \n   Consider augmenting your lore files with more LLM-generated Q&A.{Col.END}")
        else:
            print(f"\n{Col.CYAN}🚀 Dataset looks solid! Ready for fine-tuning.{Col.END}")
        print(f"═" * 45)

    except Exception as e:
        print(f"{Col.RED}❌ Failed to save dataset: {e}{Col.END}")

# --- EXECUTION ---
# save_smart_dataset(
#     all_conversations=all_conversations, 
#     char_profile=char_profile, 
#     output_path=OUTPUT_FILE,
#     model_name=CONFIG["model"]
# )


## 8 — Upload to Kaggle 

### 🚀 Option A: Automatic Upload (Kaggle API)
1. **API Key:** Ensure your `kaggle.json` file is located in `~/.kaggle/` (Linux/Mac) or `C:\Users\<User>\.kaggle\` (Windows).
2. **Metadata:** Ensure a `dataset-metadata.json` exists in your folder. You can generate one using `kaggle datasets init -p .`.
3. **Command:** Run the following command in a terminal or a code cell:
   ```bash
   # For a new dataset:
   kaggle datasets create -p .
   
   # To update an existing dataset:
   kaggle datasets version -p . -m "Updated Nayari dataset with new lore and chat logs"